In [1]:
# ===============================
# 1. Import Required Libraries
# ===============================
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Display settings
pd.set_option('display.max_columns', None)

In [2]:
# ===============================
# 2. Load Dataset
# ===============================
# Make sure diabetes.csv is in the same directory
df = pd.read_csv('diabetes.csv')

# Preview data
print("First 5 rows:")
display(df.head())

print("\nDataset Info:")
df.info()

print("\nStatistical Summary:")
display(df.describe())

First 5 rows:


,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1



Dataset Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 768 entries, 0 to 767
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Pregnancies               768 non-null    int64  
 1   Glucose                   768 non-null    int64  
 2   BloodPressure             768 non-null    int64  
 3   SkinThickness             768 non-null    int64  
 4   Insulin                   768 non-null    int64  
 5   BMI                       768 non-null    float64
 6   DiabetesPedigreeFunction  768 non-null    float64
 7   Age                       768 non-null    int64  
 8   Outcome                   768 non-null    int64  
dtypes: float64(2), int64(7)
memory usage: 54.1 KB

Statistical Summary:


,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
count,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000
mean,3.845052,120.894531,69.105469,20.536458,79.799479,31.992578,0.471876,33.240885,0.348958
std,3.369578,31.972618,19.355807,15.952218,115.244002,7.884160,0.331329,11.760232,0.476951
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.078000,21.000000,0.000000
25%,1.000000,99.000000,62.000000,0.000000,0.000000,27.300000,0.243750,24.000000,0.000000
50%,3.000000,117.000000,72.000000,23.000000,30.500000,32.000000,0.372500,29.000000,0.000000
75%,6.000000,140.250000,80.000000,32.000000,127.250000,36.600000,0.626250,41.000000,1.000000
max,17.000000,199.000000,122.000000,99.000000,846.000000,67.100000,2.420000,81.000000,1.000000


In [3]:
# ===============================
# 3. Check Missing Values
# ===============================
print("Missing values count:")
print(df.isnull().sum())

Missing values count:
Pregnancies                 0
Glucose                     0
BloodPressure               0
SkinThickness               0
Insulin                     0
BMI                         0
DiabetesPedigreeFunction    0
Age                         0
Outcome                     0
dtype: int64


In [4]:
# ===============================
# 4. Handle Invalid Zero Values
# ===============================
# In this dataset, 0 is invalid for some columns
columns_with_zero_as_missing = [
    'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI'
]

# Replace 0 with NaN
df[columns_with_zero_as_missing] = df[columns_with_zero_as_missing].replace(0, np.nan)

print("After replacing 0 with NaN:")
print(df.isnull().sum())

After replacing 0 with NaN:
Pregnancies                   0
Glucose                       5
BloodPressure                35
SkinThickness               227
Insulin                     374
BMI                          11
DiabetesPedigreeFunction      0
Age                           0
Outcome                       0
dtype: int64


In [5]:
# ===============================
# 5. Fill Missing Values
# ===============================
# Use median (robust to outliers)
# Compute medians
medians = df[columns_with_zero_as_missing].median()

# Fill missing values (NaN) with the median
df[columns_with_zero_as_missing] = df[columns_with_zero_as_missing].fillna(medians)

# Check
print("Remaining missing values:")
print(df[columns_with_zero_as_missing].isnull().sum())

Remaining missing values:
Glucose          0
BloodPressure    0
SkinThickness    0
Insulin          0
BMI              0
dtype: int64


In [6]:
# ===============================
# 6.Full Preprocessing + Feature Engineering
# ===============================

columns_zero_as_nan = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
epsilon = 1e-5  # small value to avoid division by zero

for col in columns_zero_as_nan:
    # Replace invalid 0s with NaN and fill with median
    df[col] = df[col].replace(0, np.nan).fillna(df[col].median())

# ===============================
# Categorical Features (safe)
# ===============================
# Convert pd.cut() to object type to avoid Categorical fillna errors

# BMI Category
df['BMI_Category'] = pd.cut(
    df['BMI'], bins=[-1,18.5,25,30,float('inf')],
    labels=['Underweight','Normal','Overweight','Obese']
).astype(object).fillna('Unknown')

# Age Group
df['Age_Group'] = pd.cut(
    df['Age'], bins=[0,30,40,50,60,float('inf')],
    labels=['<30','30-40','40-50','50-60','60+']
).astype(object).fillna('Unknown')

# Glucose Level
df['Glucose_Level'] = pd.cut(
    df['Glucose'], bins=[-1,100,125,float('inf')],
    labels=['Normal','Prediabetes','Diabetes']
).astype(object).fillna('Unknown')

# ===============================
# Interaction / Ratio Features
# ===============================
df['BMI_Age'] = df['BMI'] * df['Age']  # BMI multiplied by Age
df['Glucose_BMI'] = df['Glucose'] * df['BMI']  # Glucose multiplied by BMI
df['Insulin_Glucose_Ratio'] = df['Insulin'] / (df['Glucose'] + epsilon)  # Insulin efficiency
df['Pregnancy_Age_Ratio'] = (df['Pregnancies'] / (df['Age'] + 1)).clip(upper=0.5)  # capped ratio
df['Pregnancy_Risk'] = pd.cut(
    df['Pregnancy_Age_Ratio'], bins=[0,0.1,0.3,0.5],
    labels=['Low','Medium','High']
).astype(object).fillna('Unknown')
df['Insulin_log'] = np.log1p(df['Insulin'])  # log transform to reduce skewness

# ===============================
# Encode categorical features for ML
# ===============================
for col in ['BMI_Category','Age_Group','Glucose_Level','Pregnancy_Risk']:
    df[col] = pd.factorize(df[col])[0]  # convert categories to integer codes

# ===============================
# Final Cleanup
# ===============================
df = df.fillna(df.median(numeric_only=True))  # ensure no missing values remain

# ===============================
# Verification
# ===============================
print("Missing values after full feature engineering:")
print(df.isnull().sum())
display(df.head())

Missing values after full feature engineering:
Pregnancies                 0
Glucose                     0
BloodPressure               0
SkinThickness               0
Insulin                     0
BMI                         0
DiabetesPedigreeFunction    0
Age                         0
Outcome                     0
BMI_Category                0
Age_Group                   0
Glucose_Level               0
BMI_Age                     0
Glucose_BMI                 0
Insulin_Glucose_Ratio       0
Pregnancy_Age_Ratio         0
Pregnancy_Risk              0
Insulin_log                 0
dtype: int64


,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome,BMI_Category,Age_Group,Glucose_Level,BMI_Age,Glucose_BMI,Insulin_Glucose_Ratio,Pregnancy_Age_Ratio,Pregnancy_Risk,Insulin_log
0,6,148.0,72.0,35.0,125.0,33.6,0.627,50,1,0,0,0,1680.0,4972.8,0.844595,0.117647,0,4.836282
1,1,85.0,66.0,29.0,125.0,26.6,0.351,31,0,1,1,1,824.6,2261.0,1.470588,0.031250,1,4.836282
2,8,183.0,64.0,29.0,125.0,23.3,0.672,32,1,2,1,0,745.6,4263.9,0.683060,0.242424,0,4.836282
3,1,89.0,66.0,23.0,94.0,28.1,0.167,21,0,1,2,1,590.1,2500.9,1.056180,0.045455,1,4.553877
4,0,137.0,40.0,35.0,168.0,43.1,2.288,33,1,0,1,0,1422.3,5904.7,1.226277,0.000000,2,5.129899


In [7]:
# ===============================
# 7. Feature & Target Separation
# ===============================
X = df.drop('Outcome', axis=1)  # Features
y = df['Outcome']               # Target

print("Feature shape:", X.shape)
print("Target shape:", y.shape)

Feature shape: (768, 17)
Target shape: (768,)


In [8]:
# ===============================
# 8. Train-Test Split
# ===============================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42,
    stratify=y
)

print("Training set:", X_train.shape)
print("Testing set:", X_test.shape)

Training set: (614, 17)
Testing set: (154, 17)


In [9]:
# ===============================
# 9. Data Normalization (Scaling)
# ===============================
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Convert back to DataFrame (optional, for readability)
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X.columns)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X.columns)

print("Scaled training data preview:")
display(X_train_scaled.head())

Scaled training data preview:


,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,BMI_Category,Age_Group,Glucose_Level,BMI_Age,Glucose_BMI,Insulin_Glucose_Ratio,Pregnancy_Age_Ratio,Pregnancy_Risk,Insulin_log
0,-0.851355,-1.056427,-0.826740,-1.918187,-1.203361,-0.769477,0.310794,-0.792169,0.634029,0.380020,0.047775,-0.981042,-1.070048,-1.299501,-0.813767,0.360220,-2.108235
1,0.356576,0.144399,0.477772,-0.229874,-1.470195,-0.417498,-0.116439,0.561034,0.634029,-0.685424,-1.125570,0.224934,-0.182996,-1.903067,0.224303,-0.948508,-3.436957
2,-0.549372,-0.556083,-1.152868,1.233330,-0.555335,0.359790,-0.764862,-0.707594,-0.703750,0.380020,1.221119,-0.482260,-0.228055,-0.468414,-0.346065,0.360220,-0.531687
3,-0.851355,0.811525,-1.315932,-0.004766,-0.161437,-0.402832,0.262314,-0.369293,0.634029,0.380020,-1.125570,-0.507687,0.236858,-0.546202,-0.898213,0.360220,0.046763
4,-1.153338,-0.889646,-0.663676,1.120776,-0.415565,1.782373,-0.337630,-0.961320,-0.703750,0.380020,0.047775,-0.235480,0.168198,-0.050292,-1.320444,1.668948,-0.307271


In [10]:
# ===============================
# 10. Final Output Check
# ===============================
print("Final datasets ready for model training:")
print("X_train:", X_train_scaled.shape)
print("X_test:", X_test_scaled.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

Final datasets ready for model training:
X_train: (614, 17)
X_test: (154, 17)
y_train: (614,)
y_test: (154,)


In [11]:
# Save cleaned dataset
df.to_csv("processed_diabetes.csv", index=False)

print("Dataset saved successfully!")

Dataset saved successfully!
